# Grok Bias

Visualizes results from auditing X's "Explain this post" feature (Section 4): stance distribution of Grok's contextual claims and the marginal effect of each guideline in the prompt provided by X. Reads from `outputs/judge_predict/`.

In [1]:
import os
os.chdir("../")

In [ ]:
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src import utils

os.environ['PATH'] = f"{os.path.expanduser('~/.TinyTeX/bin/x86_64-linux')}:{os.environ['PATH']}"

sns.set_theme(context='paper', style='ticks', font_scale=1)

In [ ]:
name = "grok_bias"
width_pt = 469
palette = sns.color_palette('husl', 5)

STANCE_PALETTE = {'for': palette[2], 'neutral': 'silver', 'against': palette[0]}
STANCE_ORDER = ['for', 'neutral', 'against']
STANCE_PRETTY = {'for': 'Pro-choice', 'neutral': 'Neutral', 'against': 'Pro-life'}

GUIDELINE_LABELS = {
    1: "Relevance",
    2: "Non-obvious",
    3: "Challenge\nmainstream",
    4: "Evidence",
}

GUIDELINE_FULL_TEXT = {
    1: "Include only context, backstory, or world events that are directly relevant and surprising, informative, educational, or entertaining.",
    2: "Avoid stating the obvious or simple reactions.",
    3: "Provide truthful and based insights, challenging mainstream narratives if necessary, but remain objective.",
    4: "Incorporate relevant scientific studies, data, or evidence to support your analysis; prioritize peer-reviewed research and be critical of sources to avoid bias.",
}

judge_dir = "outputs/judge_predict"

## Load and reshape

In [4]:
files = sorted(glob.glob(f'{judge_dir}/judge_predict*.tsv'))
dfs = []
for f in files:
    df = pd.read_csv(f, sep='\t', dtype=str, quoting=3, on_bad_lines='warn')
    eg = int(f.rsplit('exclude_guideline=', 1)[1].split('.')[0])
    df['exclude_guideline'] = eg
    dfs.append(df)
row_df = pd.concat(dfs, ignore_index=True).reset_index(drop=True)

# Long-format: one row per judged bullet
bullet_cols = ['prediction_bullet_1', 'prediction_bullet_2', 'prediction_bullet_3']
long = []
for c in bullet_cols:
    sub = row_df[[c, 'annotation', 'exclude_guideline', 'sentence_id']].copy()
    sub.columns = ['pred', 'annotation', 'exclude_guideline', 'sentence_id']
    long.append(sub)
bullet_df = pd.concat(long, ignore_index=True).dropna(subset=['pred']).reset_index(drop=True)

src_map = {'Argument_for': 'pro-choice', 'Argument_against': 'pro-life'}
bullet_df['source'] = bullet_df['annotation'].map(src_map)

print(f"Generations (rows): {len(row_df)}")
print(f"Bullets judged:     {len(bullet_df)}")
print(f"Per source:         {bullet_df['source'].value_counts().to_dict()}")
print(f"Per label:          {bullet_df['pred'].value_counts().to_dict()}")

Generations (rows): 1950
Bullets judged:     5850
Per source:         {'pro-life': 2925, 'pro-choice': 2925}
Per label:          {'neutral': 2765, 'against': 1833, 'for': 1252}


## Stance distribution by source tweet (all four guidelines)

In [5]:
baseline = bullet_df[bullet_df['exclude_guideline'] == 0]
ct = pd.crosstab(baseline['source'], baseline['pred'], normalize='index') * 100
ct = ct[STANCE_ORDER]
ct = ct.reindex(['pro-life', 'pro-choice'])

utils.latexify()
fig_width, fig_height = utils.get_fig_dim(width_pt, fraction=0.6)
fig, ax = plt.subplots(figsize=(fig_width, fig_height))

y_pos = np.arange(len(ct))
left = np.zeros(len(ct))
for label in STANCE_ORDER:
    vals = ct[label].values
    ax.barh(y_pos, vals, left=left, color=STANCE_PALETTE[label],
            edgecolor='white', linewidth=0.6, label=STANCE_PRETTY[label])
    for i, v in enumerate(vals):
        if v > 5:
            txt_color = 'black' if label == 'neutral' else 'white'
            ax.text(left[i] + v / 2, y_pos[i], f'{v:.0f}\\%',
                    ha='center', va='center', color=txt_color, fontsize=10)
    left += vals

ax.set_yticks(y_pos)
ax.set_yticklabels(['Pro-life\nposts', 'Pro-choice\nposts'])
for label in ax.get_yticklabels():
    label.set_multialignment('center')
ax.set_xlim([0, 100])
ax.set_xlabel(r"Contextual claims generated by Grok")
# ax.legend(loc='lower center', bbox_to_anchor=(0.5, 1.02), ncol=3, frameon=False)

sns.despine(ax=ax, left=True, bottom=True)
ax.set_xticks([])
ax.tick_params(axis='y', length=0)
fig.tight_layout()
fig.savefig(f'figures/{name}__stance_dist_baseline.pdf', dpi=300)
plt.close()

## Effect of adding each guideline

In [ ]:
import textwrap
from matplotlib.ticker import FuncFormatter

pooled = pd.crosstab(bullet_df['exclude_guideline'], bullet_df['pred'], normalize='index') * 100
pooled_effect = pooled.loc[0] - pooled.drop(index=0)

plot_df = (pooled_effect
           .reset_index()
           .melt(id_vars='exclude_guideline', var_name='pred', value_name='delta'))
plot_df['guideline'] = plot_df['exclude_guideline'].map(GUIDELINE_FULL_TEXT)
guideline_order = [GUIDELINE_FULL_TEXT[g] for g in [1, 2, 3, 4]]

utils.latexify()
fig_width, fig_height = utils.get_fig_dim(width_pt, fraction=0.6)

fig, (ax_text, ax) = plt.subplots(
    1, 2,
    figsize=(2 * fig_width, fig_height),
    gridspec_kw={'width_ratios': [20, 20], 'wspace': 0},
    sharey=True,
)

sns.barplot(
    data=plot_df, y='guideline', x='delta', hue='pred',
    order=guideline_order, hue_order=STANCE_ORDER, legend=False,
    palette=STANCE_PALETTE, errorbar=None, ax=ax, orient='h',
    width=0.75,
)

ax.set_ylabel('')
ax.set_xlabel(r"Percentage change in contextual claims generated by Grok")
ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{x:.0f}\\%"))
ax.set_xlim(-10, 10)
ax.set_xticks(np.arange(-10, 11, 5))
ax.set_yticklabels([])
sns.despine(ax=ax, left=True)
ax.tick_params(axis='y', length=0)

band_color = '#f5f5f5'
band_half_height = 0.50
for ax_ in (ax_text, ax):
    for y_center in (0, 2):
        ax_.axhspan(y_center - band_half_height, y_center + band_half_height,
                    color=band_color, zorder=-1)

wrap_width = 69
text_pad_x = 0.03
wrapped = [textwrap.fill(g, width=wrap_width) for g in guideline_order]
for i, label in enumerate(wrapped):
    ax_text.text(text_pad_x, i, label, ha='left', va='center', multialignment='left',
                 fontsize=8, transform=ax_text.get_yaxis_transform())
sns.despine(ax=ax_text, top=True, right=True, left=True, bottom=True)
ax_text.set_xticks([])
ax_text.set_yticks([])
ax_text.set_xlabel("Guideline added to Grok's instructions\nwhile keeping the other three fixed")

xlabel_y = -0.18
ax.xaxis.set_label_coords(0.5, xlabel_y)
ax_text.xaxis.set_label_coords(0.5, xlabel_y)

fig.tight_layout()
fig.subplots_adjust(wspace=0)
fig.savefig(f'figures/{name}__guideline_effect_pooled.pdf', dpi=300)
plt.close()

In [ ]:
import matplotlib.patches as mpatches

utils.latexify()
handles = [mpatches.Patch(facecolor=STANCE_PALETTE[s], edgecolor='none',
                          label=STANCE_PRETTY[s])
           for s in STANCE_ORDER]

fig_leg, ax_leg = plt.subplots(figsize=(fig_width, 0.3))
ax_leg.axis('off')
ax_leg.legend(handles=handles, loc='center', ncol=len(STANCE_ORDER))

fig_leg.savefig(f'figures/{name}__legend.pdf', dpi=300, bbox_inches='tight')
plt.close()